In [2]:
!pip install PyPDF2
!pip install language-tool-python
!pip install textstat
!pip install spellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 47.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for spellchecker: filename=spellchecker-0.4-py3-none-any.whl size=3966499 sha256=6fd61919059ce3bb6fb1aaaf694873815b2c58cc00860037f5f115f0042c675d
  Stored in directory: /root/.cache/pip/wheels/1c/cb/0f/5e7178a807a0648885bca7c40bdf2db39af2d9fc1adc9855c9
  Created wheel for inexactsearch: filename=inexactsearch-1.0.2-py3-none-any.whl size=7119 sha256=97221ae521b26d4ae66b47e09da4b0eb6299c1ad6c1bc8cec36e519dcdf11141
  Stored in directory: /root/.cache/pip/wheels/e9/4a/1f/f9dd8e4c4b533bf6913f71ca99e3da8e

In [4]:
!pip uninstall pyspellchecker -y
!pip install pyspellchecker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 54.2 MB/s eta 0:00:00


In [6]:
import PyPDF2
import language_tool_python
import textstat
from spellchecker import SpellChecker

# Function to extract text from PDF
def pdf_to_text_array(pdf_path):
    text_array = []
    try:
        with open(pdf_path, 'rb') as pdf_file:
            reader = PyPDF2.PdfReader(pdf_file)
            for page in reader.pages:
                text_array.append(page.extract_text() or '')  # Handle empty pages gracefully
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return []
    return " ".join(text_array)

# Function to calculate keyword match score
def calculate_keyword_match(text, keywords):
    text_lower = text.lower()
    matches = sum(1 for keyword in keywords if keyword.lower() in text_lower)
    keyword_match_score = 10 if matches >= len(keywords) else max(5, int((matches / len(keywords)) * 10))
    return keyword_match_score

# Function to check skills highlighting
def calculate_skills_highlighting(text, skills):
    highlighted_skills = [skill for skill in skills if skill.lower() in text.lower()]
    skills_highlighting_score = 10 if len(highlighted_skills) >= len(skills) else max(5, int((len(highlighted_skills) / len(skills)) * 10))
    return skills_highlighting_score

# Function to evaluate structure and formatting
def evaluate_structure_and_formatting(text):
    lines = text.split('\n')
    has_bullets = any(line.strip().startswith(('*', '-', '•')) for line in lines)
    has_sections = any(line.strip().isupper() for line in lines)  # Assuming uppercase for section headers
    structure_score = 10 if has_bullets and has_sections else 7 if has_bullets or has_sections else 5
    return structure_score

# Function to check consistency
def evaluate_consistency(text):
    consistency_score = 10 if ' ' * 2 not in text else 8  # Penalize for inconsistent spacing
    return consistency_score

# Function to evaluate CV quality
def evaluate_cv(text, keywords, skills):
    try:
        tool = language_tool_python.LanguageTool('en-US')
        matches = tool.check(text)
        grammar_score = 10 if len(matches) <= 5 else max(6, 10 - len(matches) // 2)

        spell = SpellChecker()
        words = text.split()
        misspelled = spell.unknown(words)
        spelling_score = 10 if len(misspelled) <= 3 else max(6, 10 - len(misspelled) // 2)

        readability_score = textstat.flesch_reading_ease(text)
        if readability_score > 80:
            readability_score = 10
        elif readability_score > 65:
            readability_score = 8
        elif readability_score > 50:
            readability_score = 7
        else:
            readability_score = 5

        structure_score = evaluate_structure_and_formatting(text)
        keyword_match_score = calculate_keyword_match(text, keywords)
        skills_highlighting_score = calculate_skills_highlighting(text, skills)
        consistency_score = evaluate_consistency(text)

        total_score = (grammar_score + spelling_score + readability_score + structure_score +
                       keyword_match_score + skills_highlighting_score + consistency_score) / 7
        return round(total_score, 1), grammar_score, spelling_score, readability_score, structure_score, keyword_match_score, skills_highlighting_score, consistency_score
    except Exception as e:
        print(f"Error evaluating CV: {e}")
        return None, None, None, None, None, None, None, None

# Example usage
pdf_path = '/content/11995013.pdf'
keywords = ['Arts', 'Drawing', 'Data Analysis']  # Example keywords
skills = ['Teamwork', 'Leadership', 'Project Management']  # Example skills

text = pdf_to_text_array(pdf_path)

if text:
    cv_score, grammar_score, spelling_score, readability_score, structure_score, keyword_match_score, skills_highlighting_score, consistency_score = evaluate_cv(text, keywords, skills)
    if cv_score is not None:
        print(f"CV Quality Score: {cv_score}/10")
        print(f"Grammar Score: {grammar_score}/10")
        print(f"Spelling Score: {spelling_score}/10")
        print(f"Readability Score: {readability_score}/10")
        print(f"Structure and Formatting Score: {structure_score}/10")
        print(f"Keyword Match Score: {keyword_match_score}/10")
        print(f"Skills Highlighting Score: {skills_highlighting_score}/10")
        print(f"Consistency Score: {consistency_score}/10")
    else:
        print("Error occurred during CV evaluation.")
else:
    print("Failed to extract text from the PDF.")


CV Quality Score: 6.0/10
Grammar Score: 6/10
Spelling Score: 6/10
Readability Score: 5/10
Structure and Formatting Score: 7/10
Keyword Match Score: 5/10
Skills Highlighting Score: 5/10
Consistency Score: 8/10
